<a href="https://colab.research.google.com/github/vischia/atdtf/blob/master/example_atmosphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pietro Vischia, ATDTF

ML forecasting of gridded geophysical fields (plus metrics & simple uncertainty

Focus:
- Load a small lat-lon-time dataset via xarray
- Build persistence + CNN baselines in PyTorch
- Evaluate with lat-weighted RMSE + ACC
- (Optional) add a spectral regularizer + MC-dropout uncertainty
Dataset choice:
- xarray.tutorial.open_dataset("air_temperature") is small and convenient.
  If it can't download (no internet), the code falls back to a synthetic dataset.


In [ ]:
from __future__ import annotations

In [ ]:
import math
import os
from dataclasses import dataclass
from typing import Tuple, Optional

In [ ]:
import numpy as np

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
import xarray as xr

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 1) Load data (xarray) + standardize

In [ ]:
def load_air_temperature() -> xr.Dataset:
    """
    Attempts to load xarray tutorial dataset.
    If unavailable, returns a synthetic lat-lon-time dataset with similar structure.
    """
    try:
        ds = xr.tutorial.open_dataset("air_temperature")  # downloads a small netCDF
        # Expected: variable "air" with dims (time, lat, lon)
        return ds
    except Exception as e:
        print("Could not load xarray tutorial dataset:", repr(e))
        print("Falling back to synthetic data.")
        # Synthetic dataset: advected waves + noise on a sphere-like lat-lon grid
        time = np.arange(0, 500)  # 500 timesteps
        lat = np.linspace(-90, 90, 73)   # ~2.5°
        lon = np.linspace(0, 357.5, 144) # ~2.5°
        T = np.zeros((time.size, lat.size, lon.size), dtype=np.float32)

        # Create a simple moving pattern with lat-dependent amplitude
        Lon, Lat = np.meshgrid(lon, lat)
        lat_amp = np.cos(np.deg2rad(Lat))  # bigger near equator
        for t in range(time.size):
            phase = 2 * np.pi * (Lon / 360.0 + 0.01 * t)
            wave = np.sin(phase) * lat_amp + 0.5 * np.cos(2 * phase) * (lat_amp ** 2)
            seasonal = 0.2 * np.cos(2 * np.pi * t / 100.0) * np.sin(np.deg2rad(Lat))
            T[t] = (wave + seasonal).astype(np.float32)
        T += 0.1 * np.random.randn(*T.shape).astype(np.float32)
        ds = xr.Dataset(
            {"air": (("time", "lat", "lon"), T)},
            coords={"time": time, "lat": lat, "lon": lon},
        )
        return ds

In [ ]:
ds = load_air_temperature()
print(ds)

Select variable and ensure float32

In [ ]:
da = ds["air"].astype("float32")

Train/val/test split along time

In [ ]:
n_time = da.sizes["time"]
t_train = int(0.7 * n_time)
t_val = int(0.85 * n_time)

In [ ]:
da_train = da.isel(time=slice(0, t_train))
da_val = da.isel(time=slice(t_train, t_val))
da_test = da.isel(time=slice(t_val, None))

Standardize using training mean/std

In [ ]:
train_mean = da_train.mean(("time", "lat", "lon"))
train_std = da_train.std(("time", "lat", "lon")) + 1e-6

In [ ]:
da_train_n = (da_train - train_mean) / train_std
da_val_n = (da_val - train_mean) / train_std
da_test_n = (da_test - train_mean) / train_std

## 2) Metrics: lat-weighted RMSE and ACC

In [ ]:
def lat_weights(lat: np.ndarray) -> np.ndarray:
    """Area weights proportional to cos(lat)."""
    w = np.cos(np.deg2rad(lat)).clip(min=0.0)
    w = w / (w.mean() + 1e-12)  # normalize for numerical convenience
    return w.astype(np.float32)

In [ ]:
def weighted_rmse(y: xr.DataArray, yhat: xr.DataArray, lat_name: str = "lat") -> float:
    """
    Lat-weighted RMSE over (lat, lon), averaged over time.
    Assumes y and yhat share dims (time, lat, lon).
    """
    w = xr.DataArray(lat_weights(y[lat_name].values), dims=(lat_name,), coords={lat_name: y[lat_name]})
    mse = ((y - yhat) ** 2).weighted(w).mean((lat_name, "lon")).mean("time")
    return float(np.sqrt(mse.values))

In [ ]:
def anomaly_correlation(y: xr.DataArray, yhat: xr.DataArray, climatology: xr.DataArray) -> float:
    """
    ACC over (lat, lon), averaged over time, using a provided climatology (lat, lon) or (month, lat, lon).
    For simplicity here: climatology is (lat, lon) mean over time.
    """
    y_anom = y - climatology
    yhat_anom = yhat - climatology
    w = xr.DataArray(lat_weights(y["lat"].values), dims=("lat",), coords={"lat": y["lat"]})

    # (a·b)/(||a|| ||b||) per time step, then average
    num = (y_anom * yhat_anom).weighted(w).mean(("lat", "lon"))
    den = (
        (y_anom**2).weighted(w).mean(("lat", "lon")).pipe(np.sqrt)
        * (yhat_anom**2).weighted(w).mean(("lat", "lon")).pipe(np.sqrt)
        + 1e-12
    )
    acc = (num / den).mean("time")
    return float(acc.values)

In [ ]:
clim_train = da_train_n.mean("time")  # normalized climatology

## 3) Build supervised dataset: history -> lead

In [ ]:
@dataclass(frozen=True)
class WindowConfig:
    hist: int = 2      # number of past frames as input
    lead: int = 1      # predict t+lead
    stride: int = 1

In [ ]:
CFG = WindowConfig(hist=2, lead=1, stride=1)

In [ ]:
class XarrayWindowDataset(Dataset):
    """
    Converts an xarray DataArray (time, lat, lon) into supervised samples:
      x = [t-hist+1 ... t]  (hist frames)
      y = [t+lead]          (single frame)
    """
    def __init__(self, da_norm: xr.DataArray, cfg: WindowConfig):
        self.cfg = cfg
        self.lat = da_norm["lat"].values
        self.lon = da_norm["lon"].values
        arr = da_norm.values  # (T, Y, X)
        assert arr.ndim == 3
        self.arr = arr.astype(np.float32)
        self.T, self.Y, self.X = self.arr.shape
        self.idxs = []
        t0 = cfg.hist - 1
        t1 = self.T - cfg.lead - 1
        for t in range(t0, t1 + 1, cfg.stride):
            self.idxs.append(t)
    def __len__(self) -> int:
        return len(self.idxs)
    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor]:
        t = self.idxs[i]
        x = self.arr[t - self.cfg.hist + 1 : t + 1]         # (hist, Y, X)
        y = self.arr[t + self.cfg.lead]                      # (Y, X)

        # Torch expects (C, H, W)
        x_t = torch.from_numpy(x)                            # (hist, Y, X)
        y_t = torch.from_numpy(y).unsqueeze(0)               # (1, Y, X)
        return x_t, y_t

In [ ]:
train_ds = XarrayWindowDataset(da_train_n, CFG)
val_ds = XarrayWindowDataset(da_val_n, CFG)
test_ds = XarrayWindowDataset(da_test_n, CFG)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
print("N(train/val/test):", len(train_ds), len(val_ds), len(test_ds))

## 4) Baseline: persistence (yhat = last input frame)

In [ ]:
def persistence_forecast(da_norm: xr.DataArray, cfg: WindowConfig) -> xr.DataArray:
    """
    For each valid t, predicts y(t+lead) as x(t) (the last frame in history).
    Returns yhat with dims (time, lat, lon) aligned to the target times.
    """
    hist, lead = cfg.hist, cfg.lead
    # valid target times are: [hist-1+lead ... T-1]
    target_times = da_norm["time"].isel(time=slice(hist - 1 + lead, None))
    yhat = da_norm.isel(time=slice(hist - 1, -lead))  # the "last input" aligned with target
    yhat = yhat.assign_coords(time=target_times)
    return yhat

Evaluate persistence on test

In [ ]:
y_test = da_test_n.isel(time=slice(CFG.hist - 1 + CFG.lead, None))
yhat_pers = persistence_forecast(da_test_n, CFG)
yhat_pers = yhat_pers.isel(time=slice(0, y_test.sizes["time"]))  # safety

In [ ]:
rmse_pers = weighted_rmse(y_test, yhat_pers)
acc_pers = anomaly_correlation(y_test, yhat_pers, climatology=clim_train)
print(f"Persistence | RMSE={rmse_pers:.4f} | ACC={acc_pers:.4f}")

## 5) A small CNN forecaster (fast to train)

In [ ]:
class PeriodicPad2d(nn.Module):
    """
    Periodic padding in longitude (W dimension), zero/reflect in latitude if desired.
    Here: periodic in W, and replicate in H (simple).
    """
    def __init__(self, pad: int = 1):
        super().__init__()
        self.pad = pad
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        p = self.pad
        if p == 0:
            return x
        # periodic pad in W
        x_w = torch.cat([x[..., -p:], x, x[..., :p]], dim=-1)
        # replicate pad in H
        top = x_w[..., :1, :].repeat(1, 1, p, 1)
        bot = x_w[..., -1:, :].repeat(1, 1, p, 1)
        x_hw = torch.cat([top, x_w, bot], dim=-2)
        return x_hw

In [ ]:
class SimpleGeoCNN(nn.Module):
    def __init__(self, in_ch: int, hidden: int = 64, dropout: float = 0.0):
        super().__init__()
        self.pad = PeriodicPad2d(pad=1)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=0),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=0),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden, 1, kernel_size=1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, hist, H, W)
        x = self.pad(x)     # -> (B, C, H+2, W+2)
        y = self.net(x)     # -> (B, 1, H, W)
        return y

In [ ]:
model = SimpleGeoCNN(in_ch=CFG.hist, hidden=64, dropout=0.05).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

## 6) Optional advanced loss: spectral regularizer (encourages realistic scales)

In [ ]:
def spectral_loss(yhat: torch.Tensor, y: torch.Tensor, frac: float = 0.5) -> torch.Tensor:
    """
    Compare low-frequency magnitude spectra (very lightweight proxy for "right scales").
    frac: keep this fraction of lowest frequencies along each axis.
    """
    # yhat, y: (B, 1, H, W)
    Yh = torch.fft.rfft2(yhat, norm="ortho")
    Y = torch.fft.rfft2(y, norm="ortho")
    H, W_r = Y.shape[-2], Y.shape[-1]
    h_keep = max(1, int(frac * H))
    w_keep = max(1, int(frac * W_r))
    mag_h = torch.abs(Yh[..., :h_keep, :w_keep])
    mag = torch.abs(Y[..., :h_keep, :w_keep])
    return torch.mean((mag_h - mag) ** 2)

In [ ]:
USE_SPECTRAL_REG = True
LAMBDA_SPEC = 0.05

## 7) Train for a few epochs

In [ ]:
def run_epoch(loader: DataLoader, train: bool) -> float:
    model.train(train)
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)              # (B, hist, H, W)
        y = y.to(DEVICE)              # (B, 1, H, W)
        with torch.set_grad_enabled(train):
            yhat = model(x)
            loss = torch.mean((yhat - y) ** 2)
            if USE_SPECTRAL_REG:
                loss = loss + LAMBDA_SPEC * spectral_loss(yhat, y, frac=0.5)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        total += float(loss.item()) * x.size(0)
        n += x.size(0)
    return total / max(1, n)

In [ ]:
EPOCHS = 6
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch:02d} | train MSE(+spec)={tr:.5f} | val MSE(+spec)={va:.5f}")

## 8) Evaluate on test (convert batches back to xarray)

In [ ]:
@torch.no_grad()
def predict_xarray(da_norm: xr.DataArray, cfg: WindowConfig) -> xr.DataArray:
    """
    Produces predictions aligned with the target times:
      targets: time indices [hist-1+lead ... end]
    """
    model.eval()

    # Build a dataset for the full array
    ds_full = XarrayWindowDataset(da_norm, cfg)
    loader = DataLoader(ds_full, batch_size=32, shuffle=False, num_workers=0)
    preds = []
    for x, _ in loader:
        x = x.to(DEVICE)
        yhat = model(x).cpu().numpy()   # (B, 1, H, W)
        preds.append(yhat[:, 0])
    pred = np.concatenate(preds, axis=0)  # (N, H, W)

    # Align time coordinates to the *targets*
    target_times = da_norm["time"].isel(time=slice(cfg.hist - 1 + cfg.lead, None))
    pred_da = xr.DataArray(
        pred.astype(np.float32),
        dims=("time", "lat", "lon"),
        coords={"time": target_times, "lat": da_norm["lat"], "lon": da_norm["lon"]},
        name="air_pred",
    )
    return pred_da

In [ ]:
yhat_test = predict_xarray(da_test_n, CFG)
y_test = da_test_n.isel(time=slice(CFG.hist - 1 + CFG.lead, None))
yhat_test = yhat_test.isel(time=slice(0, y_test.sizes["time"]))  # safety

In [ ]:
rmse_cnn = weighted_rmse(y_test, yhat_test)
acc_cnn = anomaly_correlation(y_test, yhat_test, climatology=clim_train)

In [ ]:
print(f"CNN model   | RMSE={rmse_cnn:.4f} | ACC={acc_cnn:.4f}")

## 9) Optional: simple uncertainty via MC-dropout


In [ ]:
@torch.no_grad()
def mc_dropout_ensemble(x: torch.Tensor, n: int = 16) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns mean and std of predictions with dropout active.
    x: (B, C, H, W)
    """
    model.train(True)  # keep dropout active
    ys = []
    for _ in range(n):
        ys.append(model(x).detach())
    Y = torch.stack(ys, dim=0)  # (n, B, 1, H, W)
    return Y.mean(dim=0), Y.std(dim=0)

Quick demo on one minibatch

In [ ]:
x0, y0 = next(iter(test_loader))
x0 = x0.to(DEVICE)
y0 = y0.to(DEVICE)

In [ ]:
mean0, std0 = mc_dropout_ensemble(x0, n=16)
mse0 = torch.mean((mean0 - y0) ** 2).item()
print(f"MC-dropout demo | batch MSE(mean)={mse0:.5f} | avg pred std={std0.mean().item():.4f}")

## 10) (Optional plotting) - kept minimal; use matplotlib if desired


In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    # Pick one time index to visualize
    idx = 0
    y_true = y_test.isel(time=idx).values
    y_pred = yhat_test.isel(time=idx).values
    err = y_pred - y_true
    plt.figure()
    plt.title("Truth (normalized)")
    plt.imshow(y_true)
    plt.colorbar()
    plt.tight_layout()
    plt.figure()
    plt.title("Prediction (normalized)")
    plt.imshow(y_pred)
    plt.colorbar()
    plt.tight_layout()
    plt.figure()
    plt.title("Error (pred - truth)")
    plt.imshow(err)
    plt.colorbar()
    plt.tight_layout()
    plt.show()